# 01 — Request subset GFS 0.25° (NCAR GDEX d084001)

Langkah checklist: **T01.4** (cek `subsetting_available`), **T07.1–T07.5** (uji 1 bulan), **T17** (request 5 tahun).

Jalankan dari atas ke bawah. Sel **Submit** dan **Download** sengaja dipisah: jalankan hanya kalau memang ingin mengirim/mengunduh.

| Bagian | Isi |
|---|---|
| 0 | Instal & import |
| 1 | Token GDEX |
| 2 | Konfigurasi request |
| 3 | Fungsi bantu |
| 4 | **CHECK** — `subsetting_available`, template, metadata |
| 5 | **BUILD** — control file uji + 5 tahun, validasi ke metadata |
| 6 | **SUBMIT** — uji 1 bulan, lalu 5 tahun |
| 7 | **STATUS** — pantau antrean (jalankan ulang 1–2×/hari) |
| 8 | **DOWNLOAD** — unduh + MANIFEST md5 |


## 0. Instal & import

In [1]:
# Sekali saja: hapus tanda # di baris bawah bila paket belum terpasang
# %pip install -q gdex-api-client python-dotenv pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import calendar, getpass, hashlib, json, os
from datetime import date, datetime, timezone
from pathlib import Path

from gdex_api_client import gdex_client as gc

# Root repo: notebook boleh ada di root atau di folder notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CTL_DIR  = ROOT / "configs"
GFS_DIR  = ROOT / "data" / "raw" / "gfs"
META_DIR = GFS_DIR / "meta"
REQ_LOG  = GFS_DIR / "requests.json"
DECISIONS = ROOT / "decisions.md"
for d in (CTL_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)

ROOT = c:\Users\Asus\Downloads\ta-solar-forecast\ta-solar-forecast


## 0b. Sakelar akses server
`RUN_REMOTE = False` (default): notebook aman di-*Run All* — tidak meminta token, tidak mengecek/submit/unduh ke GDEX, dan tidak menambah baris ke `decisions.md`. Ubah ke `True` **hanya** bila memang perlu menghubungi GDEX lagi.

In [ ]:
RUN_REMOTE = False

## 1. Token GDEX
Dibaca dari `GDEX_TOKEN` di `.env` (root repo). Kalau tidak ada (mis. di Colab), diminta lewat input tersembunyi. Token **tidak** ditulis ke file.

In [3]:
if not RUN_REMOTE:
    print("RUN_REMOTE = False → bagian ini dilewati (tidak menghubungi GDEX)")
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(ROOT / ".env")
    except ImportError:
        pass

    TOKEN = os.environ.get("GDEX_TOKEN", "").strip() or getpass.getpass("GDEX token: ").strip()
    assert TOKEN, "Token kosong"
    gc.get_authentication = lambda token_file=None: TOKEN   # klien resmi memakai token ini
    print("Token siap (", TOKEN[:4] + "…", ")")

Token siap ( ac34… )


## 2. Konfigurasi request
- Ganti `LOCATION` dengan **koordinat final T03** sebelum submit 5 tahun.
- Kode level & nama produk divalidasi ke metadata di bagian 5; kalau ada ⚠, perbaiki di sini lalu jalankan ulang bagian 5.

In [11]:
CFG = {
    "dataset": "d084001",
    "location": {"lat": -6.8899, "lon": 107.6100},  # Kampus ITB Ganesha (T03)
    "box_half_width_deg": 0.5,
    "run_hour_utc": 0,                          # hanya run 00 UTC yang dipakai
    "leads_h": [18, 21, 24, 27, 30, 33, 36, 39],
    "test_month": "2020-01",
    "years": [2016, 2017, 2018, 2019, 2020],     # tiap tahun mulai 31 Des tahun sebelumnya
    "params": {
        "avg":  {"DSWRF": ["SFC:0"], "T CDC": ["EATM:0", "LCY:0", "MCY:0", "HCY:0"]},
        "inst": {"TMP": ["HTGL:2"], "R H": ["HTGL:2"], "P WAT": ["EATM:0"], "CAPE": ["SFC:0"]},
    },
}
OVERRIDE_KEYS = ("dataset", "date", "param", "level", "product", "nlat", "slat", "wlon", "elon")

## 3. Fungsi bantu

In [5]:
def find_key(obj, key):
    """Nilai kunci pertama di JSON bersarang (skema respons API tidak seragam)."""
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        obj = list(obj.values())
    if isinstance(obj, list):
        for v in obj:
            r = find_key(v, key)
            if r is not None:
                return r
    return None

def now_utc():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

def log_decision(text):
    with open(DECISIONS, "a") as fh:
        fh.write(f"- {date.today().isoformat()} | {text}\n")

def load_requests():
    return json.loads(REQ_LOG.read_text()) if REQ_LOG.exists() else {}

def save_requests(reqs):
    REQ_LOG.write_text(json.dumps(reqs, indent=2))

def product_strings(leads):
    """Instan: '18-hour Forecast'. Rata-rata reset tiap 6 jam:
    lead%6==3 -> '3-hour Average (initial+15 to initial+18)', lead%6==0 -> '6-hour Average (...)'."""
    inst, avg = [], []
    for L in leads:
        inst.append(f"{L}-hour Forecast")
        span = 3 if L % 6 == 3 else 6
        avg.append(f"{span}-hour Average (initial+{L - span} to initial+{L})")
    return inst, avg

def date_range(label, cfg):
    """Rentang tanggal RUN. test = satu bulan; <tahun> = 31 Des (tahun-1) s.d. 30 Des tahun itu."""
    h = int(cfg["run_hour_utc"])
    if label == "test":
        y, m = map(int, cfg["test_month"].split("-"))
        return datetime(y, m, 1, h), datetime(y, m, calendar.monthrange(y, m)[1], h)
    y = int(label)
    return datetime(y - 1, 12, 31, h), datetime(y, 12, 30, h)

def bbox(cfg):
    lat, lon, d = cfg["location"]["lat"], cfg["location"]["lon"], cfg["box_half_width_deg"]
    return {"nlat": f"{lat + d:.2f}", "slat": f"{lat - d:.2f}", "wlon": f"{lon - d:.2f}", "elon": f"{lon + d:.2f}"}

def build_ctl(label, cfg, template=None):
    inst_p, avg_p = product_strings(cfg["leads_h"])
    params = {**cfg["params"]["avg"], **cfg["params"]["inst"]}
    levels = []
    for lv in params.values():
        levels += [x for x in lv if x not in levels]
    start, end = date_range(label, cfg)
    ctl = dict(template or {})
    ctl.update({
        "dataset": cfg["dataset"],
        "date": f"{start:%Y%m%d%H%M}/to/{end:%Y%m%d%H%M}",
        "param": "/".join(params),
        "level": ";".join(levels),
        "product": "/".join(inst_p + avg_p),
        **bbox(cfg),
    })
    return ctl

def ctl_to_text(ctl):
    return "\n".join(f"{k}={v}" for k, v in ctl.items()) + "\n"

def meta_text():
    files = sorted(META_DIR.glob("*.json")) + sorted(META_DIR.glob("*.ctl"))
    return "\n".join(f.read_text() for f in files)

def validate_against_meta(ctl, mtext):
    missing = [f"param {p}" for p in ctl["param"].split("/") if f'"{p}"' not in mtext]
    missing += [f"product '{p}'" for p in ctl["product"].split("/") if p not in mtext]
    missing += [f"level {lv}" for lv in ctl["level"].split(";") if lv.split(":")[0] not in mtext]
    return missing

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

LABELS_YEARS = [str(y) for y in CFG["years"]]
print("Fungsi siap.")

Fungsi siap.


## 4. CHECK — kriteria GO #1 (T01.4)
Menyimpan summary, paramsummary, metadata, dan template ke `data/raw/gfs/meta/`, lalu mencatat hasil ke `decisions.md`.

In [6]:
if not RUN_REMOTE:
    print("RUN_REMOTE = False → bagian ini dilewati (tidak menghubungi GDEX)")
else:
    ds = CFG["dataset"]
    summary = gc.get_summary(ds)
    (META_DIR / f"{ds}_summary.json").write_text(json.dumps(summary, indent=2))

    psum = gc.get_param_summary(ds)
    (META_DIR / f"{ds}_paramsummary.json").write_text(json.dumps(psum, indent=2))
    (META_DIR / f"{ds}_metadata.json").write_text(json.dumps(gc.get_metadata(ds), indent=2))

    SUBSETTING = find_key(summary, "subsetting_available")
    if SUBSETTING is None:
        SUBSETTING = find_key(psum, "subsetting_available")

    TEMPLATE_TXT = find_key(gc.get_control_file_template(ds), "template") or ""
    (META_DIR / f"{ds}_template.ctl").write_text(TEMPLATE_TXT)

    log_decision(f"GDEX {ds}: subsetting_available = {SUBSETTING} (dicek {now_utc()}) [GO kriteria #1]")
    print(f"subsetting_available = {SUBSETTING}")
    print("⚠ Kriteria GO #1 TIDAK terpenuhi." if SUBSETTING is not True else "✓ Kriteria GO #1 terpenuhi.")
    print("\n---- template ----\n" + TEMPLATE_TXT)

subsetting_available = True
✓ Kriteria GO #1 terpenuhi.

---- template ----
dataset=ds084.1
date=201609200000/to/201609200000
datetype=init
param=TMP/R H/ABS V
level=ISBL:850/700/500
#oformat=netCDF
nlat=30
slat=-25
wlon=-150
elon=-30
product=Analysis/12-hour Forecast/6-hour Forecast/18-hour Forecast
targetdir=/glade/scratch



In [7]:
if not RUN_REMOTE:
    print("RUN_REMOTE = False → bagian ini dilewati (tidak menghubungi GDEX)")
else:
    # Lihat nama param yang tersedia (bantu memperbaiki CFG bila validasi gagal)
    rows = find_key(psum, "data")
    rows = rows.get("data", rows) if isinstance(rows, dict) else rows
    want = {"DSWRF", "TCDC", "TMP", "RH", "PWAT", "CAPE"}
    for r in rows or []:
        if isinstance(r, dict) and r.get("param") in want:
            print(json.dumps(r, ensure_ascii=False))

{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_description": "Temperature", "standard_name": "air_temperature", "GCMD_uuid": "926c1b80-6c11-40eb-ae7f-f5bcfdc43fac"}
{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_description": "Temperature", "standard_name": "air_temperature", "GCMD_uuid": "926c1b80-6c11-40eb-ae7f-f5bcfdc43fac"}
{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_description": "Temperature", "standard_name": "air_temperature", "GCMD_uuid": "926c1b80-6c11-40eb-ae7f-f5bcfdc43fac"}
{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_description": "Temperature", "standard_name": "air_temperature", "GCMD_uuid": "926c1b80-6c11-40eb-ae7f-f5bcfdc43fac"}
{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_description": "Temperature", "standard_name": "air_temperature", "GCMD_uuid": "926c1b80-6c11-40eb-ae7f-f5bcfdc43fac"}
{"units": "K", "native_format": "WMO_GRIB2", "param": "TMP", "param_de

## 5. BUILD — control file (T07.2, T17.1–T17.2)
Membuat `configs/gdex_test.ctl` dan `configs/gdex_<tahun>.ctl`, mempertahankan kunci lain dari template, lalu memvalidasi ke metadata.

In [12]:
tpath = META_DIR / f"{CFG['dataset']}_template.ctl"
template = gc.read_control_file(str(tpath)) if tpath.exists() and tpath.stat().st_size else {}
template = {k: v for k, v in template.items() if k not in OVERRIDE_KEYS}

for label in ["test"] + LABELS_YEARS:
    ctl = build_ctl(label, CFG, template)
    (CTL_DIR / f"gdex_{label}.ctl").write_text(ctl_to_text(ctl))
    print(f"✓ configs/gdex_{label}.ctl   date={ctl['date']}")

MTEXT = meta_text()
MISSING = validate_against_meta(build_ctl("test", CFG, template), MTEXT) if MTEXT else ["(metadata belum ada — jalankan bagian 4)"]
if MISSING:
    print("\n⚠ Tidak ditemukan di metadata — perbaiki CFG (bagian 2) lalu jalankan ulang sel ini:")
    for m in MISSING:
        print("   -", m)
else:
    print("\n✓ Semua param, produk, dan level ditemukan di metadata. Aman untuk submit.")

print("\n---- gdex_test.ctl ----\n" + (CTL_DIR / "gdex_test.ctl").read_text())

✓ configs/gdex_test.ctl   date=202001010000/to/202001310000
✓ configs/gdex_2016.ctl   date=201512310000/to/201612300000
✓ configs/gdex_2017.ctl   date=201612310000/to/201712300000
✓ configs/gdex_2018.ctl   date=201712310000/to/201812300000
✓ configs/gdex_2019.ctl   date=201812310000/to/201912300000
✓ configs/gdex_2020.ctl   date=201912310000/to/202012300000

✓ Semua param, produk, dan level ditemukan di metadata. Aman untuk submit.

---- gdex_test.ctl ----
datetype=init
targetdir=/glade/scratch
dataset=d084001
date=202001010000/to/202001310000
param=DSWRF/T CDC/TMP/R H/P WAT/CAPE
level=SFC:0;EATM:0;LCY:0;MCY:0;HCY:0;HTGL:2
product=18-hour Forecast/21-hour Forecast/24-hour Forecast/27-hour Forecast/30-hour Forecast/33-hour Forecast/36-hour Forecast/39-hour Forecast/6-hour Average (initial+12 to initial+18)/3-hour Average (initial+18 to initial+21)/6-hour Average (initial+18 to initial+24)/3-hour Average (initial+24 to initial+27)/6-hour Average (initial+24 to initial+30)/3-hour Average 

## 6. SUBMIT (T07.3, T17.3)
`submit(labels)` menolak jalan bila validasi bagian 5 masih gagal (pakai `force=True` untuk memaksa) dan tidak mengirim ulang label yang sudah tercatat (pakai `resubmit=True`).

In [13]:
def submit(labels, force=False, resubmit=False):
    if not RUN_REMOTE:
        print(f"RUN_REMOTE = False → submit {labels} dilewati"); return
    reqs = load_requests()
    for label in labels:
        path = CTL_DIR / f"gdex_{label}.ctl"
        assert path.exists(), f"{path.name} belum ada; jalankan bagian 5."
        if label in reqs and not resubmit:
            print(f"- {label}: sudah disubmit (ID {reqs[label]['request_id']}) — lewati.")
            continue
        ctl = gc.read_control_file(str(path))
        miss = validate_against_meta(ctl, meta_text())
        if miss and not force:
            raise RuntimeError(f"{label}: {len(miss)} item tidak ada di metadata, mis. {miss[:3]}")
        resp = gc.submit_json(ctl)
        rid = find_key(resp, "request_id") or find_key(resp, "request_index") or find_key(resp, "rindex")
        print(f"- {label}: ID={rid}\n  respons: {json.dumps(resp)[:300]}")
        if rid is None:
            print("  ⚠ ID tidak terbaca otomatis — cek respons di atas.")
        reqs[label] = {"request_id": rid, "submitted": now_utc(), "date": ctl["date"], "response": resp}
        save_requests(reqs)
        log_decision(f"GDEX request {label} disubmit, ID {rid} ({now_utc()})")

In [14]:
# 6a. Uji 1 bulan
submit(["test"])

- test: ID=978420
  respons: {"status": "ok", "http_response": 200, "error_messages": [], "data": {"request_id": "978420"}, "contact": "datahelp@ucar.edu"}


In [20]:
# 6b. Lima tahun — jalankan hanya bila subsetting_available = True
submit(LABELS_YEARS)

- 2016: sudah disubmit (ID 978421) — lewati.
- 2017: sudah disubmit (ID 978422) — lewati.
- 2018: sudah disubmit (ID 978423) — lewati.
- 2019: sudah disubmit (ID 978424) — lewati.
- 2020: sudah disubmit (ID 978425) — lewati.


## 7. STATUS (T07.4, T17.4) — jalankan ulang 1–2×/hari

In [16]:
if not RUN_REMOTE:
    print("RUN_REMOTE = False → bagian ini dilewati (tidak menghubungi GDEX)")
else:
    reqs = load_requests()
    for label, r in reqs.items():
        resp = gc.get_status(r["request_id"])
        st = find_key(resp, "status") or find_key(resp, "request_status")
        r["last_status"], r["last_checked"] = st, now_utc()
        print(f"{label:>5}  ID {r['request_id']}  status={st}   (submit {r['submitted']})")
    save_requests(reqs)

 test  ID 978420  status=ok   (submit 2026-11-18 10:16 UTC)
 2016  ID 978421  status=ok   (submit 2026-11-18 10:16 UTC)
 2017  ID 978422  status=ok   (submit 2026-11-18 10:16 UTC)
 2018  ID 978423  status=ok   (submit 2026-11-18 10:16 UTC)
 2019  ID 978424  status=ok   (submit 2026-11-18 10:16 UTC)
 2020  ID 978425  status=ok   (submit 2026-11-18 10:16 UTC)


## 8. DOWNLOAD (T07.5, T17.4) + MANIFEST md5

In [22]:
def download(labels):
    if not RUN_REMOTE:
        print(f"RUN_REMOTE = False → unduh {labels} dilewati"); return
    reqs = load_requests()
    for label in labels:
        if label not in reqs:
            print(f"- {label}: belum disubmit"); continue
        out = GFS_DIR / label
        out.mkdir(parents=True, exist_ok=True)
        resp = gc.download(str(reqs[label]["request_id"]), str(out) + os.sep)
        if not find_key(resp, "web_files"):
            print(f"- {label}: belum siap ({json.dumps(resp)[:200]})"); continue
        files = sorted(p for p in out.iterdir() if p.is_file() and p.name != "MANIFEST")
        lines = [f"# request {reqs[label]['request_id']} | date={reqs[label]['date']} | unduh {now_utc()}"]
        lines += [f"{md5(p)}  {p.name}  {p.stat().st_size}" for p in files]
        (out / "MANIFEST").write_text("\n".join(lines) + "\n")
        print(f"✓ {label}: {len(files)} file, MANIFEST ditulis")

download(LABELS_YEARS)          # ganti ke LABELS_YEARS untuk 5 tahun

100.000 % Completed
✓ 2016: 1 file, MANIFEST ditulis
100.000 % Completed
✓ 2017: 1 file, MANIFEST ditulis
100.000 % Completed
✓ 2018: 1 file, MANIFEST ditulis
100.000 % Completed
✓ 2019: 1 file, MANIFEST ditulis
100.000 % Completed
✓ 2020: 1 file, MANIFEST ditulis


**Catatan:** rentang `date` kemungkinan ikut mengambil run 06/12/18 UTC. Ukurannya kecil (kotak 1°); buang run selain 00 UTC di `gfs_process.py`.

In [24]:
import tarfile

for y in LABELS_YEARS:
    ydir = GFS_DIR / y
    tars = sorted(ydir.glob("*.tar"))
    if not tars:
        print(f"{y}: tidak ada .tar (belum diunduh?)"); continue
    for t in tars:
        marker = ydir / f".extracted_{t.name}"
        if marker.exists():
            print(f"{y}: {t.name} sudah diekstrak"); continue
        with tarfile.open(t) as tf:
            tf.extractall(ydir, filter="data")
        marker.touch()
        print(f"{y}: {t.name} diekstrak")

# Cek kelengkapan run 00 UTC
for y in LABELS_YEARS:
    f = sorted((GFS_DIR / y).rglob("gfs.0p25.*.grib2"))
    r00 = [p for p in f if p.name.split(".")[2].endswith("00")]
    days = 367 if int(y) % 4 == 0 else 366      # + 31 Des tahun sebelumnya, - 31 Des
    print(f"{y}: {len(f)} file total, {len(r00)} run 00 | harapan {(days - 1) * 8}")

2016: gfs.0p25.2015123100.f018-25.2016123000.f039.grib2.tar diekstrak
2017: gfs.0p25.2016123100.f018-25.2017123000.f039.grib2.tar diekstrak
2018: gfs.0p25.2017123100.f018-25.2018123000.f039.grib2.tar diekstrak
2019: gfs.0p25.2018123100.f018-25.2019123000.f039.grib2.tar diekstrak
2020: gfs.0p25.2019123100.f018-25.2020123000.f039.grib2.tar diekstrak
2016: 11683 file total, 2924 run 00 | harapan 2928
2017: 11654 file total, 2920 run 00 | harapan 2920
2018: 11649 file total, 2920 run 00 | harapan 2920
2019: 11656 file total, 2920 run 00 | harapan 2920
2020: 11688 file total, 2928 run 00 | harapan 2928


In [25]:
import pandas as pd
y = "2016"
have = {(p.name.split(".")[2], p.name.split(".")[3]) for p in (GFS_DIR / y).rglob("gfs.0p25.*00.f*.grib2")}
inits = pd.date_range("2015-12-31", "2016-12-30", freq="D").strftime("%Y%m%d00")
leads = [f"f{L:03d}" for L in CFG["leads_h"]]
missing = [(i, L) for i in inits for L in leads if (i, L) not in have]
print(missing)

[('2016022000', 'f039'), ('2016022200', 'f027'), ('2016092600', 'f036'), ('2016100200', 'f018')]


In [26]:
retry = {"r2016a": ("201602200000", "201602220000"),
         "r2016b": ("201609260000", "201609260000"),
         "r2016c": ("201610020000", "201610020000")}
for label, (s, e) in retry.items():
    ctl = gc.read_control_file(str(CTL_DIR / "gdex_2016.ctl"))
    ctl["date"] = f"{s}/to/{e}"
    (CTL_DIR / f"gdex_{label}.ctl").write_text(ctl_to_text(ctl))
submit(list(retry), force=True)

- r2016a: ID=978998
  respons: {"status": "ok", "http_response": 200, "error_messages": [], "data": {"request_id": "978998"}, "contact": "datahelp@ucar.edu"}
- r2016b: ID=978999
  respons: {"status": "ok", "http_response": 200, "error_messages": [], "data": {"request_id": "978999"}, "contact": "datahelp@ucar.edu"}
- r2016c: ID=979000
  respons: {"status": "ok", "http_response": 200, "error_messages": [], "data": {"request_id": "979000"}, "contact": "datahelp@ucar.edu"}


In [27]:
download(list(retry))

100.000 % Completed
✓ r2016a: 1 file, MANIFEST ditulis
100.000 % Completed
✓ r2016b: 1 file, MANIFEST ditulis
100.000 % Completed
✓ r2016c: 1 file, MANIFEST ditulis


In [28]:
import tarfile, shutil

targets = ["2016022000.f039", "2016022200.f027", "2016092600.f036", "2016100200.f018"]
found = []

# 1. Ekstrak tar hasil retry
for label in retry:
    rdir = GFS_DIR / label
    for t in sorted(rdir.glob("*.tar")):
        marker = rdir / f".extracted_{t.name}"
        if not marker.exists():
            with tarfile.open(t) as tf:
                tf.extractall(rdir, filter="data")
            marker.touch()
        print(f"{label}: {t.name} diekstrak")

    # 2. Salin hanya file yang hilang ke folder 2016
    for p in rdir.rglob("gfs.0p25.*.grib2"):
        if any(x in p.name for x in targets):
            dst = GFS_DIR / "2016" / p.name
            if not dst.exists():
                shutil.copy2(p, dst)
            found.append(p.name)

print("\nDitemukan di retry:", sorted(set(found)) or "—")
print("Masih hilang:", [x for x in targets if not any(x in f for f in found)] or "—")

# 3. Hitung ulang run 00 UTC 2016
r00 = [p for p in (GFS_DIR / "2016").rglob("gfs.0p25.*.grib2") if p.name.split(".")[2].endswith("00")]
print(f"\n2016: {len(r00)} run 00 | harapan 2928")

# 4. Tambah file yang disalin ke MANIFEST 2016
man = GFS_DIR / "2016" / "MANIFEST"
with open(man, "a") as fh:
    for name in sorted(set(found)):
        p = GFS_DIR / "2016" / name
        fh.write(f"{md5(p)}  {name}  {p.stat().st_size}  # dari request ulang\n")
print("MANIFEST 2016 diperbarui.")

r2016a: gfs.0p25.2016022000.f018-25.2016022200.f039.grib2.tar diekstrak
r2016b: gfs.0p25.2016092600.f018-039.grib2.tar diekstrak
r2016c: gfs.0p25.2016100200.f021-039.grib2.tar diekstrak

Ditemukan di retry: —
Masih hilang: ['2016022000.f039', '2016022200.f027', '2016092600.f036', '2016100200.f018']

2016: 2924 run 00 | harapan 2928
MANIFEST 2016 diperbarui.
